# Pull per-game player box score logs (with dates)

**Before running:** same kernel as the other pipeline notebooks (needs `pandas`, `nba_api`). This will make one bulk API call per season (9 total) -- may take a few minutes, same as `pull_traded_stints.ipynb`.

This feeds the day-decayed box-score prior for metric #4 (the "souped-up" prior-informed RAPM): instead of anchoring a player's box-score prior on a fixed season, weight every game they've ever played by how recently it happened, so a game from yesterday counts for more than one from three months ago, and last season fades out smoothly as this season's games accumulate -- no hard season boundary.

Uses the same `PlayerGameLogs` bulk endpoint as `pull_traded_stints.ipynb` (one call per season returns every player's game-by-game line), but keeps every row with its `GAME_DATE` instead of aggregating away the per-game detail. Output: `player_game_logs.csv` in this folder -- one row per (player, game).

In [ ]:
import pandas as pd
import time
import os
from nba_api.stats.endpoints import playergamelogs

REQUEST_TIMEOUT = 60
MAX_RETRIES = 4
RETRY_BACKOFF_BASE = 2.0

SEASONS = ["2017-18","2018-19","2019-20","2020-21","2021-22","2022-23","2023-24","2024-25","2025-26"]

def season_label_year(season):
    return int(season.split("-")[0])

def pull_game_logs(season, season_type="Regular Season"):
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = playergamelogs.PlayerGameLogs(
                season_nullable=season,
                season_type_nullable=season_type,
                league_id_nullable="00",
                timeout=REQUEST_TIMEOUT,
            )
            df = resp.get_data_frames()[0]
            if df.empty:
                raise ValueError(f"Empty response for season={season!r}")
            return df
        except Exception as e:
            last_exc = e
            wait = RETRY_BACKOFF_BASE * (2 ** attempt)
            print(f"  attempt {attempt+1} failed: {e!r} -- retrying in {wait:.1f}s")
            time.sleep(wait)
    raise last_exc

print("This will make one bulk API call per season (9 total) -- may take a few minutes.")

In [ ]:
keep_cols = ["PLAYER_ID","PLAYER_NAME","TEAM_ABBREVIATION","GAME_ID","GAME_DATE","MIN",
             "FGM","FGA","FG3M","FG3A","FTM","FTA","OREB","DREB","AST","STL","BLK","TOV","PF","PTS"]

all_logs = []
for season in SEASONS:
    label = season_label_year(season)
    print(f"Pulling {season} game logs...")
    logs = pull_game_logs(season)
    logs = logs[keep_cols].copy()
    logs["Season"] = label
    print(f"  {len(logs)} game rows, {logs['PLAYER_ID'].nunique()} players")
    all_logs.append(logs)

game_logs = pd.concat(all_logs, ignore_index=True)
game_logs["GAME_DATE"] = pd.to_datetime(game_logs["GAME_DATE"])
print(f"\nTotal game rows across all seasons: {len(game_logs)}")
game_logs.head(10)

In [ ]:
out_path = os.path.join(os.getcwd(), "player_game_logs.csv")
game_logs.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

# sanity check: a player's row count for a season should roughly match their real games played,
# and dates should span roughly October-June for a normal season
check = game_logs[game_logs["Season"] == 2025]
print(f"\n2025-26 season so far: {check['GAME_DATE'].min().date()} to {check['GAME_DATE'].max().date()}, "
      f"{check['PLAYER_ID'].nunique()} players, {len(check)} game rows")